# Per-position accept rate — eagle3 / suffix / extension / anchor (+ MTP)

Reads `simulation/results/explorations/method_depth_<wl>_s8k8.json` produced by `compute_method_depth_rates.py` and `simulation/results/qwen35_mtp/posacc_mtp_<wl>.json` from the H100 MTP run.

Methods on the same plot:

* **eagle3** — standalone backbone tree, position d matched against `gt[d-1]`.
* **suffix** — standalone suffix tree from `sx.speculate(base_ctx)`, position d vs `gt[d-1]`.
* **extension** — merged tree (no-dedup), set-OR over (b, e) cells with `b + e == d`. Per-position rate at depth d.
* **anchor** — DEDUPED merged tree's subtree at the deepest eagle3-matched backbone node (or virtual root if eagle3 missed at depth 1). Position p in the subtree is matched against `gt[eg_chain + p - 1]`.
* **mtp** — Qwen3.5-MTP measurement on H100 (`SGLANG_ORACLE_VANILLA=1`).

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/home/muchwater/advance-spec/simulation/results')
EXPL = ROOT / 'explorations'
MTP_DIR = ROOT / 'qwen35_mtp'
WORKLOADS = ['bfcl_v4', 'specbench', 'swebench_verified']
RESLICE = 's8k8'

def load_method_depth(wl):
    p = EXPL / f'method_depth_{wl}_{RESLICE}.json'
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

def load_mtp(wl):
    p = MTP_DIR / f'posacc_mtp_{wl}.json'
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

def rates_from_stats(st):
    if st is None:
        return None
    seq = st['seq_accept']
    ind = st['ind_accept']
    dg  = st['depth_ge']
    ca  = st['cond_accept']
    cd  = st['cond_denom']
    n = len(seq)
    return {
        'positions':  list(range(1, n + 1)),
        'seq_rate':   [seq[i] / dg[i] if dg[i] else float('nan') for i in range(n)],
        'ind_rate':   [ind[i] / dg[i] if dg[i] else float('nan') for i in range(n)],
        'cond_rate':  [ca[i] / cd[i] if cd[i] else float('nan') for i in range(n)],
        'depth_ge':   dg,
        'cond_denom': cd,
    }

md_data = {wl: load_method_depth(wl) for wl in WORKLOADS}
for wl, d in md_data.items():
    if d is None:
        print(f'  {wl}: method_depth NOT YET present')
    else:
        m = d.get('metadata', {})
        print(f'  {wl}: {m.get("n_sequences")} seqs, {m.get("n_steps")} steps, methods={list(d["by_method"].keys())}')

mtp_data = {wl: load_mtp(wl) for wl in WORKLOADS}
for wl, d in mtp_data.items():
    if d is None:
        print(f'  {wl}: MTP NOT YET present')
    else:
        m = d.get('metadata', {})
        print(f'  {wl}: MTP {m.get("n_requests")} reqs, {m.get("n_steps")} steps')

STYLE = {
    'eagle3':    dict(color='C0', marker='o', linestyle='-'),
    'suffix':    dict(color='C1', marker='s', linestyle='-'),
    'extension': dict(color='C3', marker='D', linestyle='-', linewidth=2),
    'anchor':    dict(color='C5', marker='*', linestyle='--', linewidth=2),
    'mtp':       dict(color='C6', marker='*', linestyle='-',  linewidth=2.2, markersize=9),
}

METRICS = [('seq_rate', 'sequential'),
           ('ind_rate', 'independent'),
           ('cond_rate', 'conditional')]

def _label(key):
    return {'seq_rate':  'sequential accept rate',
            'ind_rate':  'independent accept rate',
            'cond_rate': 'conditional accept rate'}[key]

def get_method_rates(wl):
    d = md_data.get(wl)
    if d is not None:
        for m in ('eagle3', 'suffix', 'extension', 'anchor'):
            st = d['by_method'].get(m)
            r = rates_from_stats(st)
            if r is not None:
                yield m, r
    mtp_d = mtp_data.get(wl)
    if mtp_d is not None:
        st = (mtp_d.get('position_accepts', {})
                  .get('by_proposer', {})
                  .get('mtp'))
        r = rates_from_stats(st)
        if r is not None:
            yield 'mtp', r

## Full view (positions 1..32)

In [ ]:
MAX_POS = 32

fig, axes = plt.subplots(3, 3, figsize=(20, 14), sharex=True)
for ri, (key, label) in enumerate(METRICS):
    for ci, wl in enumerate(WORKLOADS):
        ax = axes[ri, ci]
        plotted = 0
        for name, r in get_method_rates(wl):
            xs = r['positions'][:MAX_POS]
            ys = r[key][:MAX_POS]
            xs_v = [x for x, y in zip(xs, ys) if y == y]
            ys_v = [y for y in ys if y == y]
            if not xs_v:
                continue
            ax.plot(xs_v, ys_v, label=name, markersize=6, **STYLE[name])
            plotted += 1
        if plotted == 0:
            ax.text(0.5, 0.5, f'{wl}: missing', ha='center', va='center',
                    transform=ax.transAxes)
            continue
        ax.set_xlabel('position')
        ax.set_ylabel(_label(key))
        ax.set_title(f'{wl} — {_label(key)}')
        ax.set_ylim(0, 1.0)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=9)
fig.suptitle(f'positions 1..{MAX_POS}', y=1.01)
plt.tight_layout()
plt.show()

## Zoomed view (positions 1..16)

In [ ]:
MAX_POS_Z = 16

fig, axes = plt.subplots(3, 3, figsize=(20, 14), sharex=True)
for ri, (key, label) in enumerate(METRICS):
    for ci, wl in enumerate(WORKLOADS):
        ax = axes[ri, ci]
        plotted = 0
        for name, r in get_method_rates(wl):
            xs = r['positions'][:MAX_POS_Z]
            ys = r[key][:MAX_POS_Z]
            xs_v = [x for x, y in zip(xs, ys) if y == y]
            ys_v = [y for y in ys if y == y]
            if not xs_v:
                continue
            ax.plot(xs_v, ys_v, label=name, markersize=6, **STYLE[name])
            plotted += 1
        if plotted == 0:
            ax.text(0.5, 0.5, f'{wl}: missing', ha='center', va='center',
                    transform=ax.transAxes)
            continue
        ax.set_xlabel('position')
        ax.set_ylabel(_label(key))
        ax.set_title(f'{wl} — {_label(key)} (≤{MAX_POS_Z})')
        ax.set_ylim(0, 1.0)
        ax.set_xticks(range(1, MAX_POS_Z + 1))
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=9)
fig.suptitle(f'positions 1..{MAX_POS_Z}', y=1.01)
plt.tight_layout()
plt.show()

## Numeric tables (positions 1..16)

In [ ]:
for wl in WORKLOADS:
    rows = []
    for name, r in get_method_rates(wl):
        for i in range(min(MAX_POS_Z, len(r['positions']))):
            rows.append({
                'method':    name,
                'pos':       r['positions'][i],
                'seq_rate':  r['seq_rate'][i],
                'ind_rate':  r['ind_rate'][i],
                'cond_rate': r['cond_rate'][i],
            })
    if not rows:
        print(f'{wl}: no data yet')
        continue
    df = pd.DataFrame(rows)
    print(f'\n=== {wl} (positions 1..{MAX_POS_Z}) ===')
    for key, label in METRICS:
        pv = df.pivot_table(index='pos', columns='method', values=key)
        print(f'\n[{_label(key)}]')
        print(pv.round(3).to_string())